In [ ]:
# This cell can be deleted in the end.
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(1, '../')

# 08 Decoupling example

In [ ]:
import pyFBS

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 3D view

In [ ]:
view3D = pyFBS.view3D(show_origin = False, show_axes = False,shape =  (1,3),title = "Overview")

In [ ]:
pos_xlsx = pyFBS.example_lab_testbench["meas"]["xlsx_decoupling"]
view3D.plot.add_text("A structure", position='upper_left', font_size=10, color="k", font="times", name="A_structure")

view3D.plot.subplot(0,0)
view3D.plot.isometric_view()

stl_dir = pyFBS.example_lab_testbench["STL"]["A"]
view3D.add_stl(stl_dir,color = "#83afd2",name = "A");

In [ ]:
df_acc_A = pd.read_excel(pos_xlsx, sheet_name='Sensors_A')
df_chn_A = pd.read_excel(pos_xlsx, sheet_name='Channels_A')
df_imp_A = pd.read_excel(pos_xlsx, sheet_name='Impacts_A')

view3D.show_acc(df_acc_A)
view3D.show_imp(df_imp_A)
view3D.show_chn(df_chn_A)

In [ ]:
view3D.plot.subplot(0,1)
view3D.plot.isometric_view()
view3D.plot.add_text("B structure", position='upper_left', font_size=10, color="k", font="times", name="B_structure")

stl_dir = pyFBS.example_lab_testbench["STL"]["B"]
view3D.add_stl(stl_dir,color = "#83afd2",name = "B");

In [ ]:
df_acc_B = pd.read_excel(pos_xlsx, sheet_name='Sensors_B')
df_chn_B = pd.read_excel(pos_xlsx, sheet_name='Channels_B')
df_imp_B = pd.read_excel(pos_xlsx, sheet_name='Impacts_B')

view3D.show_acc(df_acc_B,overwrite = False)
view3D.show_imp(df_imp_B,overwrite = False)
view3D.show_chn(df_chn_B,overwrite = False)

In [ ]:
view3D.plot.subplot(0,2)
view3D.plot.isometric_view()
view3D.plot.add_text("AB structure", position='upper_left', font_size=10, color="k", font="times", name="AB_structure");

stl_dir = pyFBS.example_lab_testbench["STL"]["AB"]
view3D.add_stl(stl_dir,color = "#83afd2",name = "AB");

In [ ]:
df_acc_AB = pd.read_excel(pos_xlsx, sheet_name='Sensors_AB')
df_chn_AB = pd.read_excel(pos_xlsx, sheet_name='Channels_AB')
df_imp_AB = pd.read_excel(pos_xlsx, sheet_name='Impacts_AB')

view3D.show_acc(df_acc_AB,overwrite = False)
view3D.show_imp(df_imp_AB,overwrite = False)
view3D.show_chn(df_chn_AB,overwrite = False)

Views can be linked rather simply.

In [ ]:
view3D.plot.link_views()
#view3D.plot.unlink_views()

## FEM model

In [ ]:
full_file_AB = pyFBS.example_lab_testbench["FEM"]["AB_full"]
ress_file_AB = pyFBS.example_lab_testbench["FEM"]["AB_rst"]

full_file_B = pyFBS.example_lab_testbench["FEM"]["B_full"]
ress_file_B = pyFBS.example_lab_testbench["FEM"]["B_rst"]

full_file_A = pyFBS.example_lab_testbench["FEM"]["A_full"]
ress_file_A = pyFBS.example_lab_testbench["FEM"]["A_rst"]

In [ ]:
MK_A = pyFBS.MK_model(ress_file_A,full_file_A,no_modes = 100,allow_pickle= False,recalculate = False)
MK_B = pyFBS.MK_model(ress_file_B,full_file_B,no_modes = 100,allow_pickle= False,recalculate = False)
MK_AB = pyFBS.MK_model(ress_file_AB,full_file_AB,no_modes = 100,allow_pickle= False,recalculate = False)

In [ ]:
df_chn_A_up = MK_A.update_locations_df(df_chn_A)
df_imp_A_up = MK_A.update_locations_df(df_imp_A)

In [ ]:
df_chn_B_up = MK_B.update_locations_df(df_chn_B)
df_imp_B_up = MK_B.update_locations_df(df_imp_B)

In [ ]:
df_chn_AB_up = MK_AB.update_locations_df(df_chn_AB)
df_imp_AB_up = MK_AB.update_locations_df(df_imp_AB)

In [ ]:
MK_A.FRF_synth(df_chn_A_up,df_imp_A_up,f_start = 0,modal_damping = 0.003)
MK_B.FRF_synth(df_chn_B_up,df_imp_B_up,f_start = 0,modal_damping = 0.003)
MK_AB.FRF_synth(df_chn_AB_up,df_imp_AB_up,f_start = 0,modal_damping = 0.003)

## VPT 

In [ ]:
df_vp = pd.read_excel(pos_xlsx, sheet_name='VP_Channels')
df_vpref = pd.read_excel(pos_xlsx, sheet_name='VP_RefChannels')

vpt_AB = pyFBS.VPT(df_chn_AB_up,df_imp_AB_up,df_vp,df_vpref)
vpt_B = pyFBS.VPT(df_chn_B_up,df_imp_B_up,df_vp,df_vpref)

In [ ]:
vpt_AB.apply_VPT(MK_AB.freq,MK_AB.FRF)
vpt_B.apply_VPT(MK_B.freq,MK_B.FRF)

#vpt.consistency([1],[1])

In [ ]:
coh_crit = pyFBS.utility.coh_on_FRF(vpt_AB.vptData)

plt.imshow(coh_crit)
plt.colorbar(shrink = 0.8)


plt.xlabel("Input DoFs")
plt.ylabel("Output DoFs")

In [ ]:
coh_crit = pyFBS.utility.coh_on_FRF(vpt_B.vptData)

plt.imshow(coh_crit)
plt.colorbar(shrink = 0.8)

plt.xlabel("Input DoFs")
plt.ylabel("Output DoFs")

In [ ]:
freq = MK_AB.freq
Y_AB = vpt_AB.vptData
Y_B = vpt_B.vptData

display(Y_AB.shape,Y_B.shape)

## Coupling

In [ ]:
from numpy.linalg import svd
from tqdm import tqdm

In [ ]:
Y_ABnB = np.zeros((2000,24+18,24+18),dtype = complex)

Y_ABnB[:,0:24,0:24] = Y_AB
Y_ABnB[:,24:,24:] =   -1*Y_B

k = 6 + 12 # Extended compatibility and equilibrium


Bu = np.zeros((k,24+18))
Bu[:k,6:6+k] = 1*np.eye(k)
Bu[:k,24:24+k] = -1*np.eye(k)

Bf = np.zeros((k,24+18))
Bf[:k,6:6+k] = 1*np.eye(k)
Bf[:k,24:24+k] = -1*np.eye(k)


#plt.figure()
#plt.imshow(Bu)
#plt.figure()
#plt.imshow(Bf)


Y_An = np.zeros_like(Y_ABnB,dtype = complex)

Y_int = Bu@Y_ABnB@Bf.T
Y_An =Y_ABnB - Y_ABnB@Bf.T@np.linalg.pinv(Y_int)@Bu@Y_ABnB

In [ ]:
arr_coup = [0,1,2,3,4,5]
Y_A_coupled = Y_An[:,arr_coup,:][:,:,arr_coup]
Y_A_ref = MK_A.FRF

In [ ]:
s1 = 0
s2 = 2

display(df_chn_AB_up.loc[[s1]])
display(df_imp_AB_up.loc[[s2]])

plt.figure(figsize = (10,6))
plt.subplot(211)
plt.semilogy(freq,np.abs(Y_A_ref[:,s1,s2]))
plt.semilogy(freq,np.abs(Y_A_coupled[:,s1,s2]))

plt.xlim(0,2000)

plt.subplot(413)
plt.plot(freq,np.angle(Y_A_ref[:,s1,s2]))
plt.plot(freq,np.angle(Y_A_coupled[:,s1,s2]))


plt.xlim(0,2000)

